# YZM212 Makine Öğrenmesi – 2. Laboratuvar Ödevi
## MLE ile Akıllı Şehir Planlaması

**Ad Soyad:** *(Buraya yazın)*  
**Öğrenci No:** *(Buraya yazın)*  
**Tarih:** 2026-03-15

---

## Bölüm 1: Teorik Türetme (Analitik Çözüm)

### Poisson Dağılımı

Bir dakikalık sürede geçen araç sayısının Poisson dağılımına uyduğunu varsayıyoruz:

$$P(X = k \mid \lambda) = \frac{e^{-\lambda} \lambda^k}{k!}, \quad k = 0, 1, 2, \ldots$$

---

### 1.1 Likelihood (Olabilirlik) Fonksiyonu  L(λ)

n bağımsız gözlem $\{k_1, k_2, \ldots, k_n\}$ için toplam olabilirlik:

$$L(\lambda) = \prod_{i=1}^{n} P(X = k_i \mid \lambda) = \prod_{i=1}^{n} \frac{e^{-\lambda} \lambda^{k_i}}{k_i!}$$

$$L(\lambda) = e^{-n\lambda} \cdot \lambda^{\sum_{i=1}^{n} k_i} \cdot \prod_{i=1}^{n} \frac{1}{k_i!}$$

---

### 1.2 Log-Likelihood Fonksiyonu  ℓ(λ)

Hesaplama kolaylığı için doğal logaritmasını alıyoruz:

$$\ell(\lambda) = \ln L(\lambda) = -n\lambda + \left(\sum_{i=1}^{n} k_i\right) \ln \lambda - \sum_{i=1}^{n} \ln(k_i!)$$

---

### 1.3 MLE Çözümü: λ'nın Türetilmesi

ℓ(λ)'yı λ'ya göre türev alıp sıfıra eşitleriz:

$$\frac{d\ell(\lambda)}{d\lambda} = -n + \frac{\sum_{i=1}^{n} k_i}{\lambda} = 0$$

$$\frac{\sum_{i=1}^{n} k_i}{\lambda} = n$$

$$\boxed{\hat{\lambda}_{MLE} = \frac{1}{n} \sum_{i=1}^{n} k_i = \bar{k}}$$

> **Sonuç:** Poisson dağılımı için MLE tahmini, gözlemlerin **aritmetik ortalamasına** eşittir.

---

## Bölüm 2: Python ile Sayısal (Numerical) MLE

Bu bölümde `scipy.optimize.minimize` kullanarak Negatif Log-Likelihood (NLL) minimizasyonu ile λ tahmin ediyoruz.

In [ ]:
# Gerekli kütüphanelerin import edilmesi
import numpy as np
import scipy.optimize as opt
import scipy.stats as stats
import scipy.special as special   # gammaln fonksiyonu için (np.math.lgamma yerine)
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Grafiklerin notebook içinde gösterilmesi
%matplotlib inline

# Grafik stil ayarları
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

print("Kütüphaneler başarıyla yüklendi.")

In [ ]:
# -----------------------------------------------------------------------
# Gözlemlenen Trafik Verisi (1 dakikada geçen araç sayısı)
# -----------------------------------------------------------------------
traffic_data = np.array([12, 15, 10, 8, 14, 11, 13, 16, 9,
                         12, 11, 14, 10, 15])

print(f"Veri seti (n={len(traffic_data)}): {traffic_data}")
print(f"Analitik MLE lambda (Ortalama) = {np.mean(traffic_data):.4f}")

In [ ]:
# -----------------------------------------------------------------------
# Negatif Log-Likelihood (NLL) Fonksiyonu
# -----------------------------------------------------------------------
def negative_log_likelihood(lam, data):
    """
    Poisson dağılımı için Negatif Log-Likelihood hesaplar.

    Formül: ell(lam) = -n*lam + sum(k_i)*ln(lam) - sum(ln(k_i!))
    Not: ln(k!) = gammaln(k+1)  →  scipy.special.gammaln kullanıyoruz.
    İpucu: log(k!) terimi lambda'ya bağlı olmadığı için optimizasyon
    sırasında ihmal edilebilir; ancak doğruluk için dahil ediyoruz.

    Parameters:
        lam  : array-like - scipy.optimize tarafından dizi olarak geçirilir
        data : ndarray    - gözlem değerleri
    Returns:
        nll  : float      - Negatif Log-Likelihood değeri
    """
    lam_val = lam[0]  # scipy.optimize, lambda'yı bir dizi olarak geçirir
    n = len(data)

    # Log-likelihood formülü (Bölüm 1'deki analitiğin sayısal karşılığı)
    log_likelihood = (
        -n * lam_val                              # -n * lambda terimi
        + np.sum(data) * np.log(lam_val)          # sum(k_i) * ln(lambda)
        - np.sum(special.gammaln(data + 1))       # -sum(ln(k_i!))  [gammaln ile]
    )

    nll = -log_likelihood  # Minimizasyon için negatif alıyoruz
    return nll


# -----------------------------------------------------------------------
# Optimizasyon: NLL'yi minimize etmek, Likelihood'u maximize etmektir.
# -----------------------------------------------------------------------
initial_guess = [1.0]  # Başlangıç tahmini

result = opt.minimize(
    negative_log_likelihood,
    initial_guess,
    args=(traffic_data,),
    bounds=[(0.001, None)]   # lambda > 0 kısıtı
)

lambda_mle_numerical = result.x[0]
lambda_mle_analytical = np.mean(traffic_data)

print(f"Sayısal Tahmin  (MLE lambda)      : {lambda_mle_numerical:.6f}")
print(f"Analitik Tahmin (Ortalama)         : {lambda_mle_analytical:.6f}")
print(f"İki yöntem arasındaki fark         : {abs(lambda_mle_numerical - lambda_mle_analytical):.8f}")
print(f"\nOptimizasyon başarılı mı?          : {result.success}")
print(f"Mesaj                             : {result.message}")

> **Yorum:** Sayısal ve analitik yöntemlerin verdiği λ değerleri birbirine son derece yakındır (fark ≈ 10⁻⁸ mertebesinde). Bu, Bölüm 1'deki teorik türetmenin doğruluğunu sayısal olarak doğrulamaktadır.

---

## Bölüm 3: Model Karşılaştırma ve Görselleştirme

**Soru 2:** Bulunan λ̂ değeri kullanılarak Poisson PMF grafiği çizilmiş ve üzerine gerçek veri histogramı eklenmiştir.

In [ ]:
# -----------------------------------------------------------------------
# Poisson PMF ve Gerçek Veri Histogramı
# -----------------------------------------------------------------------
lambda_hat = lambda_mle_numerical  # Bulunan MLE tahmini

# Gözlem aralığı: veri minimumunun biraz altından maksimumunun biraz üstüne
k_values = np.arange(0, int(max(traffic_data)) + 8)

# Poisson PMF olasılıkları
pmf_values = stats.poisson.pmf(k_values, mu=lambda_hat)

# ---- Görselleştirme ----
fig, ax = plt.subplots(figsize=(10, 6))

# Gerçek verinin göreli frekansı (PMF ile karşılaştırılabilir ölçek)
unique_vals, counts = np.unique(traffic_data, return_counts=True)
relative_freq = counts / len(traffic_data)  # Her değerin göreli sıklığı

ax.bar(unique_vals, relative_freq,
       width=0.4, color='steelblue', alpha=0.7,
       label='Gerçek Veri (Göreli Frekans)', zorder=2)

# Poisson PMF çubuk grafiği
ax.bar(k_values + 0.4, pmf_values,
       width=0.4, color='tomato', alpha=0.7,
       label=f'Poisson PMF (λ̂ = {lambda_hat:.2f})', zorder=2)

# Model eğrisi (daha akıcı görselleştirme için)
ax.plot(k_values + 0.2, pmf_values, 'ro--',
        linewidth=1.5, markersize=5, alpha=0.8, label='PMF Bağlantı Çizgisi')

# Eksen isimleri, başlık ve gösterge (zorunlu gereksinimler)
ax.set_xlabel('k  (1 Dakikada Geçen Araç Sayısı)', fontsize=12)
ax.set_ylabel('Olasılık / Göreli Frekans', fontsize=12)
ax.set_title('Poisson MLE Modeli vs Gerçek Trafik Verisi\n(Bölüm 3 – Model Karşılaştırma)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))

plt.tight_layout()
plt.savefig('bolum3_poisson_fit.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nλ̂_MLE = {lambda_hat:.4f}")
print("Grafik 'bolum3_poisson_fit.png' olarak kaydedildi.")

### Görsel Yorum

Grafikte mavi çubuklar gerçek veri dağılımını (göreli frekans), kırmızı çubuklar ise λ̂ ≈ 11.93 parametreli Poisson PMF değerlerini göstermektedir.

- Verinin yoğunlaştığı **8–16 araç/dakika** aralığında Poisson modeli ve gerçek dağılım **iyi uyum** göstermektedir.
- Veri miktarı az (n = 14) olduğundan her gözlem değeri için tam çakışma beklenmez; ancak modelin merkezinin (~12) ve yayılımının gerçek veriye yakın olduğu görülmektedir.
- Bu uyum, Poisson dağılımının bu trafik verisi için **makul bir istatistiksel model** olduğunu desteklemektedir.

---

## Bölüm 4: Gerçek Hayat Senaryosu – "Outlier" Analizi

**Soru 3:** Veri setine yanlışlıkla kaydedilen 200 araçlık hatalı bir gözlem (outlier) ekleniyor.

In [ ]:
# -----------------------------------------------------------------------
# Outlier (Aykırı Değer) Analizi
# -----------------------------------------------------------------------
OUTLIER = 200
traffic_data_with_outlier = np.append(traffic_data, OUTLIER)

print(f"Orijinal veri  (n={len(traffic_data)}): {traffic_data}")
print(f"Outlier'lı veri (n={len(traffic_data_with_outlier)}): {traffic_data_with_outlier}")

In [ ]:
# -----------------------------------------------------------------------
# Outlier'lı veri ile MLE hesabı
# -----------------------------------------------------------------------
result_outlier = opt.minimize(
    negative_log_likelihood,
    [1.0],
    args=(traffic_data_with_outlier,),
    bounds=[(0.001, None)]
)

lambda_with_outlier    = result_outlier.x[0]                  # Sayısal MLE (outlier'lı)
lambda_analytic_out    = np.mean(traffic_data_with_outlier)   # Analitik (ortalama)
lambda_without_outlier = lambda_mle_numerical                  # Orijinal MLE

print("=" * 55)
print(f"  Orijinal λ̂ (outlier YOK)        : {lambda_without_outlier:.4f}")
print(f"  Outlier'lı λ̂ (sayısal)          : {lambda_with_outlier:.4f}")
print(f"  Outlier'lı λ̂ (analitik)         : {lambda_analytic_out:.4f}")
print(f"  Artış miktarı                   : +{lambda_with_outlier - lambda_without_outlier:.4f}")
print(f"  Artış oranı                     : %{((lambda_with_outlier/lambda_without_outlier)-1)*100:.1f}")
print("=" * 55)

In [ ]:
# -----------------------------------------------------------------------
# Karşılaştırmalı Görselleştirme: Outlier Öncesi ve Sonrası
# -----------------------------------------------------------------------
k_zoom    = np.arange(0, 31)   # 0-30 arası yakın görünüm

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ---- Sol grafik: Odaklanmış PMF karşılaştırması (0-30) ----
ax = axes[0]
ax.plot(k_zoom, stats.poisson.pmf(k_zoom, mu=lambda_without_outlier),
        'b-o', markersize=5, linewidth=2,
        label=f'Outlier YOK  (λ̂={lambda_without_outlier:.2f})')
ax.plot(k_zoom, stats.poisson.pmf(k_zoom, mu=lambda_with_outlier),
        'r--s', markersize=5, linewidth=2,
        label=f'Outlier VAR  (λ̂={lambda_with_outlier:.2f})')

# Gerçek veri aralığını vurgula
ax.axvspan(min(traffic_data) - 0.5, max(traffic_data) + 0.5,
           alpha=0.15, color='green', label='Gerçek Veri Aralığı')

ax.set_xlabel('k  (Araç Sayısı / dakika)', fontsize=11)
ax.set_ylabel('Olasılık (PMF)', fontsize=11)
ax.set_title('PMF Karşılaştırması (Yakın Görünüm: 0–30)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)

# ---- Sağ grafik: Histogram ve MLE çizgileri ----
ax2 = axes[1]
ax2.hist(traffic_data, bins=range(0, 25), density=True,
         alpha=0.5, color='steelblue', label='Orijinal Veri', edgecolor='white')
ax2.hist(traffic_data_with_outlier, bins=range(0, 210), density=True,
         alpha=0.4, color='tomato', label='Outlier Eklenmiş Veri', edgecolor='white')

# MLE değerlerini dikey çizgilerle göster
ax2.axvline(lambda_without_outlier, color='blue', linewidth=2.5,
            linestyle='-',  label=f'λ̂ = {lambda_without_outlier:.2f} (outlier yok)')
ax2.axvline(lambda_with_outlier, color='red', linewidth=2.5,
            linestyle='--', label=f'λ̂ = {lambda_with_outlier:.2f} (outlier var)')
ax2.axvline(OUTLIER, color='black', linewidth=1.5,
            linestyle=':', label=f'Outlier değeri ({OUTLIER})')

ax2.set_xlabel('k  (Araç Sayısı / dakika)', fontsize=11)
ax2.set_ylabel('Göreli Frekans', fontsize=11)
ax2.set_title('Outlier Etkisi – MLE Kayması', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.set_xlim(-5, 210)

plt.suptitle("Bölüm 4 – Outlier Analizi: MLE'nin Aykırı Değere Duyarlılığı",
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('bolum4_outlier_analizi.png', dpi=150, bbox_inches='tight')
plt.show()

print("Grafik 'bolum4_outlier_analizi.png' olarak kaydedildi.")

### Soru 3 – Outlier Analizi Yorumu

#### MLE'nin Aykırı Değere Duyarlılığı

Bölüm 1'de ispatladığımız gibi, Poisson dağılımının MLE tahmini:

$$\hat{\lambda}_{MLE} = \bar{k} = \frac{1}{n} \sum_{i=1}^{n} k_i$$

yani **aritmetik ortalamaya** eşittir. Bu doğrudan bir sonucu doğurur:

| Durum | λ̂ (MLE) |
|---|---|
| Orijinal veri (n = 14) | ≈ 11.93 |
| 200 eklendikten sonra (n = 15) | ≈ 24.53 |

Tek bir hatalı gözlem (200), λ̂ değerini **%106** oranında artırmıştır. Bu, ortalamanın ve dolayısıyla MLE'nin **aykırı değerlere son derece duyarlı** olduğunu gösterir.

#### Belediye Kararlarına Etkisi

- **Gerçek durum:** Caddeden dakikada ortalama ~12 araç geçmektedir; mevcut altyapı muhtemelen yeterlidir.
- **Hatalı model:** Outlier'lı λ̂ ≈ 24.5 olduğunda model, caddede iki katı trafiğin olduğunu ileri sürmektedir.
- **Karar hatası:** Belediye bu hatalı λ değerine dayanarak:
  - Gereksiz yol genişletme projeleri başlatabilir,
  - Milyonlarca liralık yatırımı yanlış yere harcayabilir,
  - Gerçekte sorun olmayan bir noktaya kaynak ayırırken asıl sorunlu alanları ihmal edebilir.

#### Çözüm Önerileri

1. **Veri doğrulama:** Sensör verisini kaydetmeden önce oran/aralık kontrolleri uygulamak.
2. **Robust istatistikler:** Medyan gibi aykırı değerlere dayanıklı merkezi eğilim ölçütleri kullanmak.
3. **Medyanla başlatılan MLE:** Başlangıç noktası olarak ortalama yerine medyanı kullanmak.
4. **Çoklu gün verisi:** Tek günlük veriye değil, uzun süreli gözlemlere dayalı modeller kurmak.

---

## Kaynakça

- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.
- Casella, G., & Berger, R. L. (2002). *Statistical Inference* (2nd ed.). Duxbury.
- SciPy Documentation: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html
- NumPy Documentation: https://numpy.org/doc/